# LSTM Fake Review Detection — Yelp
**Model:** Bidirectional LSTM (128 units) + GloVe 300d + GlobalMaxPooling1D

**Training data:** YelpZip + YelpChi + YelpNYC — balanced to ~180k reviews (all fake + equal genuine sample), from `yelp_split.csv`

**Labels:** -1 / Y = fake (1) &nbsp;|&nbsp; 1 / N = genuine (0)

**Class imbalance:** handled via balanced sampling (1:1 ratio)

**Tuned for maximum macro F1:**
- **`MAX_LEN = 250` (not 150):** ~21% of Yelp reviews exceed 150 words; 250 cuts truncation to ~8% — the single biggest accuracy lever. *(Serving + eval code updated to 250 to match.)*
- **GloVe 300d (not 100d):** richer pre-trained embeddings — one of the largest quality levers for an LSTM.
- **BiLSTM 128 (not 64):** more capacity, controlled by SpatialDropout1D(0.3) + Dropout(0.5) + L2.
- GloVe fine-tuned on task, GlobalMaxPooling1D, early stopping (val_loss, patience=3), ReduceLROnPlateau.

---
### Instructions
1. Go to **Runtime → Change runtime type → A100 GPU** (or T4) before running.
2. Run `prepare_yelp_split.ipynb` first — it generates `yelp_split.csv` on your Google Drive.
3. Update the path in **cell 3** if your `yelp_split.csv` is in a subfolder.
4. Run all cells in order. GloVe (~822 MB) downloads automatically if not already present.
5. Download `yelp_fake_lstm.onnx`, `yelp_fake_lstm_tokenizer.json`, and `yelp_fake_lstm_threshold.json` and place them in `data/yelp_fake_lstm_onnx/`.

In [ ]:
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

In [ ]:
!pip install -q tf2onnx

In [ ]:
# Download GloVe 6B embeddings — extract the 300d file (richer than 100d) — skip if present
import os
if not os.path.exists('glove.6B.300d.txt'):
    !wget -q --show-progress http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -q glove.6B.zip glove.6B.300d.txt
    print('GloVe 300d downloaded.')
else:
    print('GloVe 300d already present — skipping download.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── UPDATE THIS PATH if your split file is in a different location ──
SPLIT_CSV = '/content/drive/MyDrive/yelp_split.csv'

In [ ]:
import random
import json
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM, Dense,
    Dropout, SpatialDropout1D, GlobalMaxPooling1D,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────
# MAX_LEN 250, not 150: ~21% of Yelp reviews exceed 150 words; 250 cuts that
# to ~8%. NOTE: serving (score_reviews.py YELP_LSTM_MAX_LEN) and the eval
# scripts must use the SAME value — they have been updated to 250.
MAX_LEN       = 250
VOCAB_SIZE    = 50_000
EMBED_DIM     = 300       # GloVe 300d (richer than 100d) — must match glove.6B.300d.txt
LSTM_UNITS    = 128       # 64 → 128: more capacity (BiLSTM → 256-dim pooled output)
BATCH_SIZE    = 256
EPOCHS        = 15
LEARNING_RATE = 5e-4
GLOVE_PATH    = 'glove.6B.300d.txt'
# ─────────────────────────────────────────────────────────────────

In [ ]:
import pandas as pd

df_split = pd.read_csv(SPLIT_CSV)

X_train = df_split[df_split['split'] == 'train']['text'].tolist()
y_train = df_split[df_split['split'] == 'train']['label'].tolist()
X_val   = df_split[df_split['split'] == 'val']['text'].tolist()
y_val   = df_split[df_split['split'] == 'val']['label'].tolist()
X_test  = df_split[df_split['split'] == 'test']['text'].tolist()
y_test  = df_split[df_split['split'] == 'test']['label'].tolist()

print(f'Train: {len(X_train):,}   Val: {len(X_val):,}   Test: {len(X_test):,}')

In [ ]:
print('=== Split sanity check ===')
print(df_split['split'].value_counts())
print()
print(pd.crosstab(df_split['split'], df_split['label']))
print()
print(pd.crosstab(df_split['split'], df_split['label'], normalize='index').round(3))
print()
print('Missing values:')
print(df_split[['text', 'label', 'split']].isna().sum())
print()
print('Label values:', sorted(df_split['label'].unique()))

In [ ]:
print('Building tokenizer on training set ...')
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)
print(f'  Vocabulary size: {len(tokenizer.word_index):,} unique words')

def encode(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_train_seq = encode(X_train)
X_val_seq   = encode(X_val)
X_test_seq  = encode(X_test)

y_train_arr = np.array(y_train)
y_val_arr   = np.array(y_val)
y_test_arr  = np.array(y_test)

print(f'X_train_seq : {X_train_seq.shape}')
print(f'X_val_seq   : {X_val_seq.shape}')
print(f'X_test_seq  : {X_test_seq.shape}')

In [ ]:
print('Loading GloVe vectors ...')
glove_embeddings = {}
with open(GLOVE_PATH, encoding='utf-8') as f:
    for line in f:
        parts = line.split()
        word = parts[0]
        vec  = np.array(parts[1:], dtype=np.float32)
        glove_embeddings[word] = vec
print(f'  Loaded {len(glove_embeddings):,} GloVe vectors')

# Build embedding matrix — rows indexed by tokenizer word_index
embedding_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM), dtype=np.float32)
hits, misses = 0, 0
for word, idx in tokenizer.word_index.items():
    if idx >= VOCAB_SIZE:
        continue
    vec = glove_embeddings.get(word)
    if vec is not None:
        embedding_matrix[idx] = vec
        hits += 1
    else:
        misses += 1

print(f'  Coverage: {hits:,} hits / {misses:,} misses ({hits/(hits+misses):.1%})')

In [ ]:
import os, shutil

# Remove stale artifacts from any previous run
for path in [
    'yelp_fake_lstm.keras',
    'yelp_fake_lstm_savedmodel',
    'yelp_fake_lstm.onnx',
    'yelp_fake_lstm_tokenizer.json',
    'yelp_fake_lstm_threshold.json',
]:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f'Removed old folder : {path}')
    elif os.path.isfile(path):
        os.remove(path)
        print(f'Removed old file   : {path}')
print('Clean.')

In [ ]:
# Improved BiLSTM — GloVe embeddings + GlobalMaxPooling (less overfitting than stacked BiLSTM)
# input named 'input_layer' to match ONNX serving code
inputs = Input(shape=(MAX_LEN,), name='input_layer')
x = Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=EMBED_DIM,
    weights=[embedding_matrix],
    trainable=True,               # fine-tune GloVe on task
)(inputs)
x = SpatialDropout1D(0.3)(x)     # drops whole feature maps, better for sequences than Dropout
x = Bidirectional(LSTM(
    LSTM_UNITS,
    return_sequences=True,
    kernel_regularizer=l2(1e-4),
))(x)
x = GlobalMaxPooling1D()(x)       # takes max over time — more stable than second BiLSTM
x = Dense(64, activation='relu', kernel_regularizer=l2(1e-4))(x)
x = Dropout(0.5)(x)
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
]

history = model.fit(
    X_train_seq, y_train_arr,
    validation_data=(X_val_seq, y_val_arr),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks
)

In [ ]:
from sklearn.metrics import f1_score

# Threshold tuning on validation set — optimise macro F1 (both classes equally)
val_probs = model.predict(X_val_seq, batch_size=BATCH_SIZE).flatten()
best_thresh_lstm, best_val_f1 = 0.5, 0.0
for t in np.arange(0.10, 0.90, 0.01):
    f = f1_score(y_val_arr, (val_probs >= t).astype(int), average='macro', zero_division=0)
    if f > best_val_f1:
        best_val_f1 = f
        best_thresh_lstm = round(float(t), 2)
print(f'Best threshold (macro F1): {best_thresh_lstm:.2f}  (Val macro F1: {best_val_f1:.1%})')

# Official test result — threshold-tuned only, no mixed reporting
y_pred_probs = model.predict(X_test_seq, batch_size=BATCH_SIZE).flatten()
y_pred = (y_pred_probs >= best_thresh_lstm).astype(int)
print(f'\nClassification report (threshold = {best_thresh_lstm:.2f}) — official result:')
print(classification_report(y_test_arr, y_pred, target_names=['Genuine', 'Fake']))

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
import json

# Save tokenizer
with open('yelp_fake_lstm_tokenizer.json', 'w', encoding='utf-8') as f:
    f.write(tokenizer.to_json())
print('Tokenizer saved: yelp_fake_lstm_tokenizer.json')

# Save threshold — serving code must load this and apply it manually
with open('yelp_fake_lstm_threshold.json', 'w') as f:
    json.dump({'threshold': best_thresh_lstm}, f, indent=2)
print(f'Threshold saved: yelp_fake_lstm_threshold.json  (value: {best_thresh_lstm})')

# Save full Keras model (weights + architecture) — needed for CPU-only ONNX re-export
model.save('yelp_fake_lstm.keras')
print('Keras model saved: yelp_fake_lstm.keras')

In [ ]:
!pip install -q onnxruntime

import subprocess, os

# Re-export SavedModel in a subprocess with CUDA hidden so TF uses standard
# LSTM ops instead of CudnnRNNV3 (which ONNX Runtime does not support).
cpu_script = """\
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import tensorflow as tf
model = tf.keras.models.load_model('yelp_fake_lstm.keras')
model.export('yelp_fake_lstm_savedmodel')
print('CPU SavedModel export done')
"""
with open('cpu_export.py', 'w') as f:
    f.write(cpu_script)

r1 = subprocess.run(
    ['python', 'cpu_export.py'],
    capture_output=True, text=True,
    env={**os.environ, 'CUDA_VISIBLE_DEVICES': ''}
)
print(r1.stdout)
if r1.returncode != 0:
    print('STDERR:', r1.stderr)

# Convert the CPU SavedModel to ONNX
r2 = subprocess.run(
    ['python', '-m', 'tf2onnx.convert',
     '--saved-model', 'yelp_fake_lstm_savedmodel',
     '--output', 'yelp_fake_lstm.onnx',
     '--opset', '13'],
    capture_output=True, text=True
)
print(r2.stdout)
if r2.returncode != 0:
    print('STDERR:', r2.stderr)

In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession('yelp_fake_lstm.onnx', providers=['CPUExecutionProvider'])
print('ONNX inputs :', [i.name for i in sess.get_inputs()])
print('ONNX outputs:', [o.name for o in sess.get_outputs()])

sample_seq = X_test_seq[:4].astype(np.float32)
input_name = sess.get_inputs()[0].name
preds = sess.run(None, {input_name: sample_seq})[0]
print('Sample predictions:', preds.flatten())

In [ ]:
import zipfile
from google.colab import files

with zipfile.ZipFile('yelp_fake_lstm_onnx.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write('yelp_fake_lstm.onnx')
    zf.write('yelp_fake_lstm_tokenizer.json')
    zf.write('yelp_fake_lstm_threshold.json')
print('Created yelp_fake_lstm_onnx.zip')

files.download('yelp_fake_lstm_onnx.zip')

## Next steps
Extract `yelp_fake_lstm_onnx.zip` and place the files in a dedicated subfolder in your project's `data/` directory:
```
data/yelp_fake_lstm_onnx/
    yelp_fake_lstm.onnx
    yelp_fake_lstm_tokenizer.json
    yelp_fake_lstm_threshold.json   ← classification threshold tuned on validation set
```
The serving code must load `yelp_fake_lstm_threshold.json` and apply it manually — the ONNX model outputs raw probabilities.